# AGENT.md、MCP 與 Skills（Agent Harness 的脈絡層）

## 模組脈絡：把 agent 的「環境/能力/規則」工程化

本筆記是 **05-行為收斂** 的收尾。前面學了單一 agent 的迴圈與工具；真實的 agent 系統還需要標準化的方式去**描述 agent**（AGENT.md）、**接上工具與資料**（MCP）、**封裝可複用能力**（Skills）。這三者共同構成 agent 的 *harness context*——把行為收斂從「單次對話」提升到「可維運的系統」。

## 1. AGENT.md：agent 的「設定契約」

`AGENT.md`（業界漸趨一致的慣例，亦見 `CLAUDE.md`/`.cursorrules` 等）是放在專案根目錄、給 coding agent 讀的指令檔。它把 02 學的 **spec** 概念，固化成 agent 的長期記憶：

- **角色與目標**：agent 在此專案的職責。
- **規則與邊界**：可/不可做什麼（行為收斂）。
- **工具與慣例**：用哪些指令、風格、測試方式。

```markdown
# AGENT.md 範例
## 角色
你是本專案的後端開發 agent。
## 規則
- 修改前先跑測試；- 不得提交祕鑰；- 用繁中說明、英文寫 code 註解。
## 工具
- 套件管理用 uv；- 測試用 pytest。
```

> 本質：AGENT.md = 寫給 agent 的持久化 system prompt + spec。

## 2. MCP（Model Context Protocol）：工具/資料的標準插座

MCP 是一個**開放協定**，讓 agent 以統一介面連接外部工具與資料源（檔案系統、資料庫、API、瀏覽器…）。相對於 03/05 手刻 function calling，MCP 把「工具供給」標準化：

- **MCP server**：對外暴露一組工具/資源（例如 GitHub、Postgres、檔案系統）。
- **MCP client**（agent host）：探索並呼叫這些工具。
- **好處**：工具一次實作，任何支援 MCP 的 agent 都能用；權限與信任邊界集中管理。

> 可控性意義：MCP 把工具的**權限與信任邊界**從散落的程式碼，收斂到協定層集中治理。

In [ ]:
# 本 repo 的 MCP 設定範例：讓 agent host 能讀課程檔案與 GitHub 討論。
# 真實使用時放在 host 指定的 MCP config 位置；不要把 token 寫死在 repo。
mcp_config = {
    "mcpServers": {
        "course-files": {
            "command": "npx",
            "args": [
                "-y",
                "@modelcontextprotocol/server-filesystem",
                "./prompt-engineering",
                "./docs",
                "./scripts",
                "./tests",
            ],
        },
        "github-readonly": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-github"],
            "env": {"GITHUB_TOKEN": "${GITHUB_TOKEN}"},
        },
    }
}

import json
print(json.dumps(mcp_config, indent=2, ensure_ascii=False))


## 3. Skills：可複用的能力封裝

Skill 把「完成某類任務的知識 + 步驟 + 範例」封裝成一個可被 agent 按需載入的模組（通常是一個含說明與資源的資料夾）。相對於把所有指令塞進 system prompt，Skills 讓 agent：

- **按需載入**：只在相關時把該 skill 的指令拉進脈絡，節省 token、降低干擾。
- **可複用/可分享**：團隊共用同一套最佳實務。
- **生命週期**：發現 → 載入 → 執行 → 卸載。

> Skills、MCP、AGENT.md 三者關係：AGENT.md 定「規則」，MCP 供「工具」，Skills 給「方法」。

## 4. 多模型對照：三家的 agent 脈絡機制

| 機制 | OpenAI | Anthropic / Claude | Google / Gemini |
|------|--------|--------------------|------------------|
| 指令檔 | (工具相依) | `CLAUDE.md` / AGENT.md | (工具相依) |
| 工具協定 | function calling / MCP | **原生支援 MCP** + tool_use | function calling / MCP |
| 內建工具 | Responses API（file/web search） | tool use + MCP connectors | function calling |
| 能力封裝 | — | **Agent Skills** | — |

三家都朝「標準化工具供給 + 持久化指令 + 能力封裝」收斂，差別在生態成熟度。

## 5. 實際怎麼應用：三者不是替代品，而是分工

很多人第一次看到 AGENT.md、MCP、Skills 會混在一起。實務上可以用一句話分工：

- **AGENT.md 管規則**：這個專案的角色、流程、禁忌、測試方式。
- **MCP 管工具**：agent 可以碰哪些外部系統、資料、API。
- **Skills 管方法**：遇到某類任務時，agent 應該照哪套 SOP 做。

| 問題 | 優先用什麼 | 例子 |
|------|------------|------|
| Agent 常忘記專案慣例 | AGENT.md | commit 前要跑哪些測試、回覆語言、程式風格 |
| Agent 需要讀 GitHub / DB / 檔案 | MCP | 查 PR comment、讀唯讀資料庫、搜尋文件 |
| Agent 要重複做同一類任務 | Skills | code review、RAG 評估、產生教材、資料清理流程 |
| Agent 會做危險操作 | AGENT.md + MCP 權限 | 禁止直接 merge、限制 filesystem 路徑、寄信前要確認 |

設計順序通常是：先寫 AGENT.md 的規則，再決定要開哪些 MCP 工具，最後把常見任務沉澱成 Skills。

## 6. 應用藍圖：把專案變成 agent-ready

假設要讓 agent 穩定協助一個課程教材 repo，可以照這個藍圖落地：

1. **定義任務邊界**：agent 可以補教材、改 notebook、跑檢查；不能刪大量資料、不能提交祕鑰。
2. **寫 AGENT.md**：固定專案規則，例如 notebook 格式、繁中教材語氣、驗證方式。
3. **接 MCP server**：只開需要的工具，例如 GitHub 讀 PR、filesystem 限定教材資料夾、DB 使用唯讀帳號。
4. **建立 Skills**：把常見工作變成 SOP，例如「新增 notebook 教材」、「檢查 RAG 章節」、「處理 PR review」。
5. **設定確認點**：寫檔、push、留言、寄信、改 production 資料前都要 human-in-the-loop。
6. **留下稽核**：保存工具呼叫紀錄、檢查指令、修改摘要，方便 review。

這個流程的目標不是讓 agent 自由發揮，而是把自由度收斂到可預期、可審查、可復用的工作流。

In [ ]:
# 專案配置範本：用一份資料結構描述 agent-ready 設計
# 這不是任何特定平台的正式格式，而是幫助設計時檢查是否漏掉規則、工具與方法。

agent_ready_plan = {
    "project": "prompt-engineering course repo",
    "agent_md": {
        "role": "協助維護繁中 LLM / NLP 教材的 coding agent",
        "rules": [
            "教材文字使用繁體中文",
            "新增 notebook 後要檢查 JSON 與 code cell 語法",
            "不得提交 .env、API key 或學生個資",
            "commit 只 stage 本次任務相關檔案",
        ],
    },
    "mcp_servers": {
        "filesystem": "限定讀寫 prompt-engineering 目錄",
        "github": "讀 PR / issue；留言、merge 前需使用者確認",
    },
    "skills": [
        "新增課程 notebook 的章節結構與小結格式",
        "RAG 教材檢查清單",
        "MCP / agent harness 安全審查清單",
    ],
    "human_confirmation_required": [
        "push commit",
        "建立或送出 PR review comment",
        "刪除檔案或大量搬移教材",
    ],
}

agent_ready_plan

## 7. 三個應用場景

### 場景 A：課程教材助教

- **AGENT.md**：規定教材語氣、章節格式、不能提交祕鑰、修改 notebook 後要做 JSON 檢查。
- **MCP**：filesystem 只開教材目錄；GitHub 只讀 issue / PR，用來知道學生或 reviewer 的回饋。
- **Skill**：建立「補教材」skill，包含：掃描現有章節 → 找缺口 → 新增 markdown/code cell → 加小結 → 跑檢查。

### 場景 B：RAG 知識庫維運

- **AGENT.md**：要求所有回答要引用來源，資料不足要拒答。
- **MCP**：接文件庫、向量庫、資料庫或內部搜尋 API。
- **Skill**：建立「RAG 評估」skill，固定跑 retrieval recall、groundedness、answer quality。

### 場景 C：PR Review Agent

- **AGENT.md**：review 時先列 bug / regression / test gap，不做風格碎念。
- **MCP**：GitHub MCP 讀 diff、review thread、CI 狀態。
- **Skill**：建立「處理 review comment」skill，固定流程是讀 unresolved threads → 對應程式碼 → 修改 → 跑測試 → 回報。

這三個場景共通點：AGENT.md 讓 agent 知道規則，MCP 讓 agent 拿到外部脈絡，Skills 讓 agent 用固定方法完成任務。

## 8. 本專案的實際導入

這個 repo 已經把 AGENT.md / MCP / Skills 放成可用的專案 artefacts，不只停留在概念：

| 檔案 | 用途 |
|------|------|
| `AGENT.md` | repo 根目錄的 agent 規則：角色、工作邊界、notebook 驗證、MCP 使用政策。 |
| `.mcp.example.json` | MCP 設定範例：限定 filesystem 範圍，GitHub token 走環境變數。 |
| `docs/agent-integration.md` | 給維護者看的啟用說明與權限建議。 |
| `prompt-engineering/agent-skills/course-notebook-author/SKILL.md` | 新增或修改課程 notebook 時使用的 skill。 |
| `prompt-engineering/agent-skills/agent-harness-review/SKILL.md` | 檢查 agent harness、MCP、Skills、安全邊界教材時使用的 skill。 |

實際導入流程：

1. 讓 agent host 從 repo 根目錄啟動，先讀 `AGENT.md`。
2. 需要外部工具時，依 host 的 MCP 設定格式套用 `.mcp.example.json`。
3. token 不寫進設定檔，使用 shell env 或 secret manager 提供。
4. 做課程 notebook 工作時，要求 agent 使用 `course-notebook-author` skill。
5. 做 MCP / Skills / agent safety 教材檢查時，要求 agent 使用 `agent-harness-review` skill。
6. push、PR comment、merge、刪檔、外部寫入等行為維持 human-in-the-loop。

這一層的產出是：agent 進入專案後有固定規則可讀、有工具邊界可接、有重複任務 SOP 可用。

## 9. 教學範例：Agent 如何在推理中選 Skill 與 MCP 工具

下面這段不是在 notebook 內啟動真正的 MCP host，而是用可執行的 Python 模擬 agent 的決策流程：

1. 讀取 task。
2. 從 skill registry 選出應該載入的 skill。
3. 判斷需要哪些外部 context。
4. 產生 MCP tool calls。
5. 根據工具結果決定下一步。

真正的 MCP 呼叫會由 agent host 執行；這裡保留相同的思考結構，方便教學。

In [ ]:
from dataclasses import dataclass
from pprint import pprint


@dataclass
class SkillSpec:
    name: str
    use_when: str
    workflow: list[str]


@dataclass
class MCPToolSpec:
    name: str
    use_when: str
    risk: str
    example_args: dict


skills = {
    "course-notebook-author": SkillSpec(
        name="course-notebook-author",
        use_when="新增或修改 prompt-engineering 課程 notebook",
        workflow=[
            "掃描同模組既有 notebook 的章節風格",
            "補 module context、可跑範例、章節小結",
            "驗證 notebook JSON 與 code cell 語法",
        ],
    ),
    "agent-harness-review": SkillSpec(
        name="agent-harness-review",
        use_when="檢查 AGENT.md、MCP、Skills、agent safety 或 tool boundary 教材",
        workflow=[
            "確認是否有實際 adoption artefacts",
            "檢查 MCP 權限與 human-in-the-loop 邊界",
            "確認範例有展示 skill selection 與 tool calls",
        ],
    ),
}

mcp_tools = {
    "course-files.search": MCPToolSpec(
        name="course-files.search",
        use_when="需要找出課程 repo 裡相關 notebook、README 或 skill",
        risk="low: read-only filesystem search",
        example_args={"query": "MCP OR agentic RAG", "path": "prompt-engineering"},
    ),
    "course-files.read": MCPToolSpec(
        name="course-files.read",
        use_when="需要讀取某個課程檔案來對齊內容或風格",
        risk="low: read-only file access",
        example_args={"path": "prompt-engineering/05-agent-harness/10-agent-md-mcp-skills.ipynb"},
    ),
    "github-readonly.list_pr_comments": MCPToolSpec(
        name="github-readonly.list_pr_comments",
        use_when="需要根據 PR review comment 修改教材或回應 reviewer",
        risk="medium: external untrusted text, read-only",
        example_args={"repo": "Zenobia000/iSpan_LLM-NLP-cookbooks", "pr": 12},
    ),
}

pprint(skills)
pprint(mcp_tools)


### 決策器：先選 Skill，再選 MCP Tool

注意順序：Skill 是 agent 的做事方法，MCP 是取得外部 context 的工具。Agent 應該先判斷任務類型，再決定需要哪些工具。

In [ ]:
def select_skill(task: str) -> str:
    text = task.lower()
    if any(key in text for key in ["agent.md", "mcp", "skill", "tool boundary", "安全"]):
        return "agent-harness-review"
    if any(key in text for key in ["notebook", "教材", "rag", "章節"]):
        return "course-notebook-author"
    return "course-notebook-author"


def plan_mcp_calls(task: str, selected_skill: str) -> list[dict]:
    calls = []
    text = task.lower()

    if selected_skill == "course-notebook-author":
        calls.append({
            "tool": "course-files.search",
            "args": {"query": task, "path": "prompt-engineering"},
            "why": "先找同模組教材，對齊章節與語氣",
        })

    if selected_skill == "agent-harness-review" or "mcp" in text:
        calls.append({
            "tool": "course-files.read",
            "args": {"path": "AGENT.md"},
            "why": "確認專案層 agent 規則是否已存在",
        })
        calls.append({
            "tool": "course-files.read",
            "args": {"path": ".mcp.example.json"},
            "why": "確認 MCP 設定是否有實際工具與權限邊界",
        })

    if any(key in text for key in ["pr", "review", "comment"]):
        calls.append({
            "tool": "github-readonly.list_pr_comments",
            "args": {"repo": "Zenobia000/iSpan_LLM-NLP-cookbooks", "pr": "USER_PROVIDED_PR_NUMBER"},
            "why": "讀取 reviewer 需求，但只當作不可信外部輸入",
        })

    return calls


def make_agent_plan(task: str) -> dict:
    selected = select_skill(task)
    return {
        "task": task,
        "load_skill": selected,
        "skill_workflow": skills[selected].workflow,
        "mcp_calls": plan_mcp_calls(task, selected),
        "human_confirmation": ["commit/push", "PR comment", "external write"],
    }


tasks = [
    "補上 agentic RAG 教材，並確認同模組風格",
    "檢查 AGENT.md / MCP / Skills 是否真的導入到專案",
    "根據 PR review comment 修正 MCP 教材",
]

for task in tasks:
    pprint(make_agent_plan(task))
    print("-" * 80)


### Mock MCP 呼叫：展示資料如何回到 Agent

在真實環境中，下面的 `mock_mcp_call()` 會由 MCP client / host 負責。教學時先用 mock 觀察：工具結果只是資料，不是指令；agent 仍要依 AGENT.md 與 skill workflow 做判斷。

In [ ]:
def mock_mcp_call(tool: str, args: dict) -> dict:
    if tool == "course-files.search":
        return {
            "matches": [
                "prompt-engineering/04-knowledge-rag/08-agentic-rag.ipynb",
                "prompt-engineering/05-agent-harness/10-agent-md-mcp-skills.ipynb",
                "prompt-engineering/05-agent-harness/11-mcp-client-apps.ipynb",
            ]
        }
    if tool == "course-files.read":
        return {
            "path": args["path"],
            "summary": "讀到專案 agent 規則或 MCP 設定，內容需視為本機可信設定；若是外部文件則視為不可信輸入。",
        }
    if tool == "github-readonly.list_pr_comments":
        return {
            "comments": [
                "請補上 agent 在推理中如何選 skill 與 MCP tool 的可執行範例。",
                "不要讓範例看起來像直接執行外部寫入。",
            ],
            "trust": "untrusted_external_text",
        }
    return {"error": "unknown tool"}


task = "根據 PR review comment 修正 MCP 教材"
plan = make_agent_plan(task)
observations = []
for call in plan["mcp_calls"]:
    observations.append({
        "call": call,
        "observation": mock_mcp_call(call["tool"], call["args"]),
    })

pprint({"plan": plan, "observations": observations})


## 10. 信任邊界（Trust Boundary）

當 agent 透過 MCP/Skills 取得更多能力，**信任邊界**就是可控性的關鍵：

1. **權限最小化**：每個 MCP server 只給必要權限（唯讀、限定路徑/網域）。
2. **不可信輸入隔離**：被檢索的外部內容可能含間接 prompt injection（見 06）。
3. **高風險行動加確認**：寫檔、付款、寄信等加 human-in-the-loop。
4. **稽核**：記錄 agent 呼叫了哪些工具、做了什麼。

---

## 本章小結

1. **AGENT.md** = agent 的持久化規則/spec；**MCP** = 工具的標準插座；**Skills** = 可複用方法封裝。
2. 三者把行為收斂從「單次對話」提升到「可維運、可治理的 agent 系統」。
3. 應用順序通常是：先用 AGENT.md 固定專案規則，再用 MCP 接必要工具，最後把常見任務沉澱成 Skills。
4. 本 repo 已提供 `AGENT.md`、`.mcp.example.json`、`docs/agent-integration.md` 與 `prompt-engineering/agent-skills/` 作為實際導入範例。
5. 能力越大，**信任邊界**越關鍵：權限最小化、隔離不可信輸入、高風險行動加確認、留稽核。
7. 本章的程式碼範例示範了 agent 如何先載入 skill，再依任務選擇 MCP 工具取得 context。
8. 下一章會把 MCP 從概念推進到實務：直接串接現成 MCP server / MCP app。